# PolyMath Benchmark Evaluation

Evaluate models on the [Qwen/PolyMath](https://huggingface.co/datasets/Qwen/PolyMath) benchmark with checkpointing support.
PolyMath is a multilingual math reasoning benchmark with 18 languages, 4 difficulty levels, and 500 problems per language.
Results are saved incrementally to `output_dir` so runs can be interrupted and resumed.

## Setup

In [ ]:
import sys, os

# When running in Colab, clone the repo and add lib/ to the path
if "google.colab" in sys.modules:
    if not os.path.exists("cs224n-final-project"):
        !git clone https://github.com/anujjamwal/cs224n-final-project.git
    !cd cs224n-final-project && git pull --recurse-submodules
    sys.path.insert(0, "cs224n-final-project/lib")
    sys.path.insert(0, "cs224n-final-project/scripts")
    sys.path.insert(0, "cs224n-final-project")
    !pip install math-verify trl

    from google.colab import drive

    drive.mount('/content/drive')
    DATA_DIR="/content/drive/My Drive"

else:
    # Local: notebook lives inside lib/ already
    sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
    DATA_DIR="."

In [ ]:
!pip install "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
!pip install --no-dependencies --upgrade flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from eval import load_polymath, run_eval, summarize_results

## Configuration

Edit the cell below to configure the evaluation.

In [ ]:
# ---- Edit these ----
BATCH_SIZE = 1
MAX_NEW_TOKENS = 2048
LANGUAGE = "en"      # Language config (e.g. "en", "zh", "fr", "de")
LEVELS = ["low"]        # e.g. ["top", "high"] to filter by difficulty
DTYPE = torch.bfloat16
MODEL_PATH = "anujjamwal/OpenMath-Nemotron-1.5B-PruneAware"
OUTPUT_DIR = os.path.join(DATA_DIR, f"cs224n/eval/polymath-en-prune-aware-07a3478f3873ec6d8f2abb38c5073abe15033181-{MAX_NEW_TOKENS}")

## Load Model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=DTYPE, device_map="auto", trust_remote_code=True, attn_implementation="flash_attention_2")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model.eval()
print(f"Model loaded: {MODEL_PATH} ({sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params)")

generate_kwargs = {
    'processing_class': tokenizer,
    "temperature": 0.7,
    "top_p": 0.9,
    "do_sample": True,
}

## Load Dataset

In [ ]:
problems = load_polymath(LANGUAGE, levels=LEVELS)
print(f"Loaded {len(problems)} PolyMath problems (language={LANGUAGE})")

## Warmup

In [ ]:
# Preview
for p in problems[:3]:
    print(f"  [{p.id}] level={p.metadata['level']}")
    print(f"    {p.problem[:80]}...")
    print(f"    expected: {p.expected_answer}")

In [ ]:
warmup = ["What is 1+1 ?", "what is 2+2 ?"]
prompts = [
    [
        {"role": "system", "content": "Solve the following math problem. Make sure to put the answer (and only answer) inside \\boxed{}."},
        {"role": "user", "content": p},
    ]
    for p in warmup
]
for prompt in prompts:
    inp = tokenizer.apply_chat_template(
        prompt,
        add_generation_prompt=True,
        padding=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        out = model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS, **generate_kwargs)
    decoded = tokenizer.decode(out, skip_special_tokens=False)
    print(decoded)

## Run Evaluation

Results are checkpointed after each batch. If you interrupt and re-run this cell,
it will skip already-evaluated problems.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

results = run_eval(
    model, tokenizer, problems, OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    generate_kwargs=generate_kwargs,
)

## Summary

In [ ]:
import pandas as pd

summary = summarize_results(results)
print(f"Overall: {summary['correct']}/{summary['total']} ({summary['accuracy']:.1%})")
print()

# By mode
if summary.get("by_mode"):
    print("By mode:")
    for mode, stats in summary["by_mode"].items():
        print(f"  {mode}: {stats['correct']}/{stats['total']} ({stats['accuracy']:.1%})")
    print()

# By level (difficulty)
if summary.get("by_level"):
    print("By difficulty level:")
    level_order = ["top", "high", "medium", "low"]
    for level in level_order:
        if level in summary["by_level"]:
            stats = summary["by_level"][level]
            print(f"  {level}: {stats['correct']}/{stats['total']} ({stats['accuracy']:.1%})")

## Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy by difficulty level
if summary.get("by_level"):
    level_order = ["top", "high", "medium", "low"]
    levels = [l for l in level_order if l in summary["by_level"]]
    accs = [summary["by_level"][l]["accuracy"] * 100 for l in levels]
    labels = [f"{l}\n({'easiest' if l == 'top' else 'hardest' if l == 'low' else ''})" for l in levels]
    ax = axes[0]
    bars = ax.bar(labels, accs, color=["#55A868", "#4C72B0", "#DD8452", "#C44E52"])
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"PolyMath ({LANGUAGE}) — Accuracy by Difficulty Level")
    ax.set_ylim(0, 100)
    for bar, v in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

# Accuracy by mode
if summary.get("by_mode"):
    modes = list(summary["by_mode"].keys())
    accs = [summary["by_mode"][m]["accuracy"] * 100 for m in modes]
    ax = axes[1]
    bars = ax.bar(modes, accs, color=["#4C72B0", "#DD8452", "#55A868"][:len(modes)])
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"PolyMath ({LANGUAGE}) — Accuracy by Mode")
    ax.set_ylim(0, 100)
    for bar, v in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, v + 1, f"{v:.1f}%", ha="center", fontsize=9)
    plt.setp(ax.get_xticklabels(), rotation=15, ha="right")

plt.tight_layout()
plt.show()

## Inspect Individual Results

In [ ]:
# Load results as a DataFrame for exploration
df = pd.DataFrame([{
    "id": r.problem_id,
    "mode": r.mode,
    "correct": r.correct,
    "expected": r.expected,
    "predicted": r.predicted,
    "generated_tokens": r.generated_tokens,
    "wall_time": r.wall_time,
    "level": r.metadata.get("level"),
    "language": r.metadata.get("language"),
} for r in results])

print(f"Total results: {len(df)}")
print(f"\nIncorrect predictions (sample):")
display(df[~df["correct"]].head(10))

In [ ]:
df.to_json(os.path.join(OUTPUT_DIR, "polymath_output.json"), orient="records", lines=True)